In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [3]:
import numpy as np
import pandas as pd
from sklearn.metrics import ndcg_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import pickle

In [4]:
from utils.train_eval_utils import train_eval
from utils.Encoder_model import make_Encoder_model
from utils.preprocess import Dataset_for_transformer, preprocess_data_4_TN
from utils.loss_mask_utils import create_mask, Cross_Entropy_point, ListNet_Loss, Combined_Loss

In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

file_path_train = r'/home/aletovv/data/train_split_web30.pkl'
file_path_test = r'/home/aletovv/data/test_split_web30.pkl'

train_data = preprocess_data_4_TN(file_path_train, num_docs=700, is_shuffle=True, device = device)
val_data = preprocess_data_4_TN(file_path_test, num_docs=700, is_shuffle=True, device = device)

preprocess is done
preprocess is done


In [7]:
X_train = []
y_train = []
i = 0
for key in train_data.keys():
    if train_data[key][0].shape[0] >= 5:
        X_train.append(train_data[key][0])
        y_train.append(train_data[key][1])
        i += 1


X_train, y_train = np.array(X_train, dtype=object), np.array(y_train, dtype=object)


In [8]:
X_val = []
y_val = []
for key in val_data.keys():
    if val_data[key][0].shape[0] > 1:
        X_val.append(val_data[key][0])
        y_val.append(val_data[key][1])
        i += 1


In [9]:
from pytorch_tabnet.abstract_model import TabModel

In [10]:
from scipy.special import softmax
from pytorch_tabnet.utils import SparsePredictDataset, PredictDataset, filter_weights
from pytorch_tabnet.abstract_model import TabModel
from pytorch_tabnet.multiclass_utils import infer_output_dim, check_output_dim
from torch.utils.data import DataLoader
import scipy


class TabNetRanker(TabModel):
    def __post_init__(self):
        super(TabNetRanker, self).__post_init__()
        # self._task = 'classification'
        # self._default_loss = ListNet_Loss(distribution='polynomial', degree=2)
        self._default_loss = Cross_Entropy_point()
        self._default_metric = 'accuracy'


    def prepare_target(self, y):
        return y
    
    def stack_batches(self, ys, scores):
        y_true = [y.cpu().numpy() if isinstance(y, torch.Tensor) else y for y in ys]

        # scores могут быть tensor или numpy
        final_scores = []
        for s in scores:
            if isinstance(s, torch.Tensor):
                final_scores.append(s.cpu().numpy())
            else:
                final_scores.append(s)
        
        return y_true, final_scores


    def compute_loss(self, y_pred, y_true):
        return self.loss_fn(y_pred, y_true.long())

In [11]:
ranker = TabNetRanker(output_dim=5)

/home/aletovv/anaconda3/envs/ltr/lib/python3.10/site-packages/pytorch_tabnet/abstract_model.py:86: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


In [12]:
ranker.fit(
    X_train=X_train, 
    y_train=y_train,
    eval_set=[(X_val, y_val)],
    eval_name=["eval"],
    eval_metric=["ndcg@5", "ndcg@10", "ndcg@all", "mrr", "err"],
    batch_size=1,
    virtual_batch_size=10000,
)

epoch 0  | loss: 1.04956 | eval_ndcg@5: 0.45905 | eval_ndcg@10: 0.46506 | eval_ndcg@all: 0.74383 | eval_mrr: 0.79296 | eval_err: 0.30467 |  0:07:16s


epoch 1  | loss: 1.03892 | eval_ndcg@5: 0.41206 | eval_ndcg@10: 0.44241 | eval_ndcg@all: 0.73553 | eval_mrr: 0.75186 | eval_err: 0.27323 |  0:14:15s


epoch 2  | loss: 1.03512 | eval_ndcg@5: 0.47231 | eval_ndcg@10: 0.47918 | eval_ndcg@all: 0.75133 | eval_mrr: 0.79781 | eval_err: 0.30745 |  0:21:03s


epoch 3  | loss: 1.03056 | eval_ndcg@5: 0.44093 | eval_ndcg@10: 0.45343 | eval_ndcg@all: 0.73771 | eval_mrr: 0.77068 | eval_err: 0.29296 |  0:27:46s


epoch 4  | loss: 1.02744 | eval_ndcg@5: 0.47733 | eval_ndcg@10: 0.47777 | eval_ndcg@all: 0.75067 | eval_mrr: 0.79368 | eval_err: 0.32837 |  0:34:32s


KeyboardInterrupt: 